# Dataset 2D — Versión para llenado humano desde planos

**Objetivo**: producir un subset del dataset original con sólo las features que un operador puede llenar **manualmente desde un plano 2D** (sin requerir análisis de CAD 3D). Esto permite ampliar el dataset incluyendo pedidos históricos que sólo tienen dibujos 2D, que son la mayoría.

**Punto de partida**: `line_items_step1.parquet` (output de `01_cleaning.ipynb`, ya con outliers removidos, categóricas tratadas, target log y duplicados eliminados).

**Pipeline**:
1. Cargar `line_items_step1.parquet`.
2. Inventario del estado actual.
3. Eliminar features (10 columnas justificadas abajo).
4. Validar estado final.
5. Guardar `line_items_2d.parquet` + CSV preview.

**Criterio de eliminación**: una feature se elimina si **al menos una** de las siguientes aplica:
- Sólo es computable desde un mesh CAD 3D (no del plano 2D).
- Es redundante con otra feature para CatBoost (el modelo elegido).

**Nota**: este notebook NO entrena modelos. Sólo prepara el dataset. El modelado va en `04_modeling_2d.ipynb`.

> **Sobre `Heat Treatment` y `Post Production`**: aunque tienen 60-76% NaN, **mantenemos** estas columnas. Según el negocio, NaN en estas columnas significa **"no se aplica"** (no es un dato faltante) — es decir, son una categoría legítima ("None") con señal real para el precio.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 200)

INPUT_PATH       = Path('line_items_step1.parquet')
OUTPUT_PATH      = Path('line_items_2d.parquet')
CSV_PREVIEW_PATH = Path('line_items_2d_preview.csv')

## 1. Cargar dataset

In [2]:
df = pd.read_parquet(INPUT_PATH)
print(f'Shape: {df.shape}')
print(f'\nColumnas:')
for c in df.columns:
    print(f'  - {c}  ({df[c].dtype})')

Shape: (386, 24)

Columnas:
  - Material  (category)
  - Technology  (category)
  - Tolerance  (float64)
  - Post Production  (category)
  - Finishing  (category)
  - Heat Treatment  (category)
  - Quantity  (int64)
  - Unit Price (USD)  (float64)
  - part_volume_cm3  (float64)
  - stock_volume_cm3  (float64)
  - material_removal_pct  (float64)
  - stock_dimensions_mm.x  (float64)
  - stock_dimensions_mm.y  (float64)
  - stock_dimensions_mm.z  (float64)
  - faces_qty  (float64)
  - aspect_ratio  (float64)
  - thin_ratio  (float64)
  - plane_ratio  (float64)
  - cyl_ratio  (float64)
  - complex_ratio  (float64)
  - total_holes  (float64)
  - part_type  (category)
  - log_unit_price  (float64)
  - log_quantity  (float64)


## 2. Estado actual

Inventario rápido para tener referencia antes/después: missingness, n_unique, dtype.

In [3]:
overview = pd.DataFrame({
    'dtype': df.dtypes.astype(str),
    'n_missing': df.isna().sum(),
    'pct_missing': (df.isna().mean() * 100).round(1),
    'n_unique': df.nunique(dropna=True),
})
overview.sort_values('pct_missing', ascending=False)

,dtype,n_missing,pct_missing,n_unique
Heat Treatment,category,293,75.9,4
Post Production,category,229,59.3,5
Tolerance,float64,124,32.1,9
part_type,category,5,1.3,4
Material,category,0,0.0,8
Finishing,category,0,0.0,5
Quantity,int64,0,0.0,36
Technology,category,0,0.0,5
Unit Price (USD),float64,0,0.0,257
part_volume_cm3,float64,0,0.0,309


## 3. Features a eliminar

### Grupo 1 — CAD-only (8 features)
Sólo se obtienen analizando el mesh 3D (volúmenes, conteos de caras, ratios de tipos de superficie). **Imposible llenar desde un plano 2D**.

| Feature | Por qué CAD-only |
|---|---|
| `part_volume_cm3` | Volumen real de la pieza (post-mecanizado), requiere modelo sólido. |
| `material_removal_pct` | `(stock − part) / stock`, depende de `part_volume_cm3`. |
| `faces_qty` | Conteo de caras del mesh; no se puede contar desde un 2D. |
| `aspect_ratio` | Calculable de dimensiones — pero la corr con target es ~0.002. |
| `thin_ratio` | % de superficie con paredes delgadas, sólo del mesh. |
| `plane_ratio` | % de superficie plana, sólo del mesh. |
| `cyl_ratio` | % de superficie cilíndrica, sólo del mesh. |
| `complex_ratio` | % de superficie compleja, sólo del mesh. |

### Grupo 2 — `stock_volume_cm3` (1 feature)
Aunque tiene buena correlación con el target (0.51), lo eliminamos por decisión: el modelo se queda con las dimensiones x/y/z individuales y deberá aprender por sí mismo qué hacer con ellas.

### Grupo 3 — Redundante para CatBoost (1 feature)

| Feature | Por qué dropear |
|---|---|
| `log_quantity` | Para árboles (CatBoost), `Quantity` y `log_quantity` se splittean equivalentemente — los thresholds son monótonos y `log` no cambia el orden. La info ya está en `Quantity`. (Era útil sólo para baselines lineales tipo Ridge.) |

### Conservadas (decisión revisada)

`Heat Treatment` y `Post Production` se **mantienen**: el NaN en estas columnas no es dato faltante sino "no se aplica tratamiento / no aplica post-producción". El modelado las usa con la etiqueta `'None'` para reflejar esta semántica.

**Total drops**: 8 + 1 + 1 = **10 features**.

In [4]:
drop_cad_only = [
    'part_volume_cm3',
    'material_removal_pct',
    'faces_qty',
    'aspect_ratio',
    'thin_ratio',
    'plane_ratio',
    'cyl_ratio',
    'complex_ratio',
]

drop_stock_volume = [
    'stock_volume_cm3',
]

drop_redundant = [
    'log_quantity',
]

# Heat Treatment y Post Production se MANTIENEN: NaN = "no aplica", no es missing.

to_drop = drop_cad_only + drop_stock_volume + drop_redundant
missing_cols = [c for c in to_drop if c not in df.columns]
if missing_cols:
    print(f'Aviso: columnas no encontradas en el input (ya eliminadas?): {missing_cols}')

to_drop = [c for c in to_drop if c in df.columns]
df_2d = df.drop(columns=to_drop)

print(f'Columnas eliminadas ({len(to_drop)}):')
print(f'  CAD-only ({len(drop_cad_only)}): {drop_cad_only}')
print(f'  Stock volume ({len(drop_stock_volume)}): {drop_stock_volume}')
print(f'  Redundante ({len(drop_redundant)}): {drop_redundant}')
print(f'\nShape antes: {df.shape}  ->  después: {df_2d.shape}')

Columnas eliminadas (10):
  CAD-only (8): ['part_volume_cm3', 'material_removal_pct', 'faces_qty', 'aspect_ratio', 'thin_ratio', 'plane_ratio', 'cyl_ratio', 'complex_ratio']
  Stock volume (1): ['stock_volume_cm3']
  Redundante (1): ['log_quantity']

Shape antes: (386, 24)  ->  después: (386, 14)


## 4. Estado final

Las 12 features que quedan son llenables manualmente desde un plano 2D + el order/PO:

**Categóricas (6)** — desde el title block del plano:
- `Material`, `Technology`, `Finishing`, `part_type`
- `Heat Treatment`, `Post Production` (NaN = "no aplica")

**Numéricas (6)**:
- `Tolerance` (title block), `Quantity` (PO), `total_holes` (contable visualmente)
- `stock_dimensions_mm.x/y/z` (bounding box del plano)

Más los targets: `Unit Price (USD)` y `log_unit_price`.

In [5]:
overview_final = pd.DataFrame({
    'dtype': df_2d.dtypes.astype(str),
    'n_missing': df_2d.isna().sum(),
    'pct_missing': (df_2d.isna().mean() * 100).round(1),
    'n_unique': df_2d.nunique(dropna=True),
})
print(f'Shape final: {df_2d.shape}')
print(f'Columnas restantes: {df_2d.shape[1]}')
print()
overview_final.sort_values('pct_missing', ascending=False)

Shape final: (386, 14)
Columnas restantes: 14



,dtype,n_missing,pct_missing,n_unique
Heat Treatment,category,293,75.9,4
Post Production,category,229,59.3,5
Tolerance,float64,124,32.1,9
part_type,category,5,1.3,4
Technology,category,0,0.0,5
Material,category,0,0.0,8
Finishing,category,0,0.0,5
Quantity,int64,0,0.0,36
stock_dimensions_mm.x,float64,0,0.0,69
Unit Price (USD),float64,0,0.0,257


In [6]:
df_2d.head(5)

,Material,Technology,Tolerance,Post Production,Finishing,Heat Treatment,Quantity,Unit Price (USD),stock_dimensions_mm.x,stock_dimensions_mm.y,stock_dimensions_mm.z,total_holes,part_type,log_unit_price
0,Other_rare,CNC Multiaxis,0.050,Anodizing II,As Machined,NaN,10,58.0000,57.1,12.7,101.6,8.0,rotational,4.060443
1,6061-T6 Aluminum,CNC Multiaxis,0.025,Other_rare,As Machined,NaN,10,174.0000,88.9,12.7,108.0,8.0,rotational,5.159055
2,Other,3D Printing SLS,0.127,NaN,Smooth Machining,NaN,1,103.5108,101.6,304.8,12.7,0.0,complex_3d,4.639676
3,Other,3D Printing SLS,0.250,NaN,As Machined,NaN,20,100.6355,101.6,304.8,12.7,0.0,complex_3d,4.611505
4,Other_rare,CNC Multiaxis,0.020,NaN,Fine Machining,NaN,1,737.0600,50.8,76.2,133.3,1.0,complex_3d,6.602669


## 5. Guardar

- **Parquet** (preserva dtypes, incluyendo `category`).
- **CSV preview** para inspección manual.

In [7]:
df_2d.to_parquet(OUTPUT_PATH, index=False)
df_2d.to_csv(CSV_PREVIEW_PATH, index=False)

print(f'Parquet:  {OUTPUT_PATH}  ({df_2d.shape[0]} filas, {df_2d.shape[1]} columnas)')
print(f'CSV:      {CSV_PREVIEW_PATH}')

Parquet:  line_items_2d.parquet  (386 filas, 14 columnas)
CSV:      line_items_2d_preview.csv


## Próximo paso

**Estado**: dataset reducido a 12 features llenables manualmente + 2 targets.

Para validar la decisión, hay que:

1. **Reentrenar el modelo CatBoost** con `line_items_2d.parquet` y comparar contra el modelo de `02_modeling.ipynb` (que usaba 22 features con CAD). Si el desempeño no cae mucho, la decisión es ganadora porque permite ampliar el dataset.
2. **Diseñar el form humano** en el frontend (`radii-app-vue`) o en el flujo de cotización, con los 12 campos.
3. **Ampliar el dataset** llenando manualmente pedidos históricos con sólo plano 2D. Cada pedido nuevo etiquetado es una fila más para entrenar.

El siguiente notebook (`04_modeling_2d.ipynb`) hace el punto 1.